In [11]:
from collections.abc import Callable

import torch
from torch import nn
from d2l import torch as d2l

In [12]:
# Kernel & Regression Data

KernelFunction = Callable[
    [torch.Tensor], # [argument type]
    torch.Tensor,   # return type
]

def gaussian(
    x: torch.Tensor,
) -> torch.Tensor:
    return torch.exp(
        - (x.pow(2)) / 2
    )
    
def boxcar(
    x: torch.Tensor,
) -> torch.Tensor:
    return (
        torch.abs(x) <= 1.0
    )
    
def constant(
    x: torch.Tensor,
) -> torch.Tensor:
    return torch.ones_like(x)

def epanechikov(
    x: torch.Tensor,
) -> torch.Tensor:
    return torch.maximum(
        1 - torch.abs(x),
        torch.zeros_like(x),
    )
    
def target_function(
    x: torch.Tensor,
) -> torch.Tensor:
    return (
        2 * torch.sin(x) + x
    )
    
    
kernels: tuple[KernelFunction, ...] = (
    gaussian,
    boxcar,
    constant,
    epanechikov,
)

kernel_names = (
    "Gaussian",
    "Boxcar",
    "Constant",
    "Epanechikov",
)


num_train = 40

# Key: [K]
x_train, _ = torch.sort(
    torch.rand(num_train) * 5
)

# Value: [K]
y_train = (
    target_function(x_train)
    + torch.randn(num_train)
)

# Query: [Q]
x_val = torch.arange(
    0,
    5,
    0.1,
)

# Ground Truth: [Q]
y_val = target_function(
    x_val
)

print(
    "Number of Keys:",
    x_train.numel(),
)
print(
    "Number of Queries:",
    x_val.numel(),
)

print(
    "Key shape:",
    tuple(x_train.shape),
)
print(
    "Value shape:",
    tuple(y_train.shape),
)
print(
    "Query shape:",
    tuple(x_val.shape),
)

Number of Keys: 40
Number of Queries: 50
Key shape: (40,)
Value shape: (40,)
Query shape: (50,)


In [13]:
# Nadaraya-Watson Attention Pooling

def nadaraya_watson(
    x_train: torch.Tensor, # Key
    y_train: torch.Tensor, # Value
    queries: torch.Tensor, # Query
    kernel: KernelFunction,
) -> tuple[
    torch.Tensor,
    torch.Tensor,
]:

    # Distances k ~ q:
    # Key:[K, 1] - Query:[1, Q] -> [K, Q]
    distances = (
        x_train.reshape(-1, 1)   # Key  :[K, 1]
        - queries.reshape(1, -1) # Query:[1, Q]
    )
    
    # Transformation: Distances -> Similarities
    similarities = kernel(
        distances
    ).to(dtype=torch.float32)
    
    
    # Attention Weights / Normalization to each Query:
    # [K, Q] / [1, Q] -> [K, Q]
    attention_weights = (
        similarities
        / similarities.sum(  
            dim=0,
            keepdim=True,
        )
    )
    
    # 각 Query에 대해 Prediction:
    # Value:[K] @ Weight:[K, Q] = Prediction[Q]
    predictions = (
        y_train
        @ attention_weights
    )
    
    return (
        predictions,
        attention_weights,
    )

In [15]:
# Gaussian Attention Data Flow

# distances: [K, Q]
distances = (
    x_train.reshape(-1, 1) # [K, 1]
    - x_val.reshape(1, -1) # [1, Q]
)

# similarity: [K, Q]
similarities = gaussian(
    distances
)

# predictions:       [Q]
# attention_weights: [K, Q]
predictions, attention_weights = (
    nadaraya_watson(
        x_train=x_train,
        y_train=y_train,
        queries=x_val,
        kernel=gaussian,
    )
)

# 각 Query에 배정된 모든 Key Weight의 합
weight_sums = attention_weights.sum(
    dim=0,
)

print(
    "Distance shape:",
    tuple(distances.shape),
)
print(
    "Similarity shape:",
    tuple(similarities.shape),
)
print(
    "Attention Weight shape:",
    tuple(attention_weights.shape),
)
print(
    "Prediction shape:",
    tuple(predictions.shape),
)

print(
    "\nFirst five Weight sums:",
    weight_sums[:5],
)

assert distances.shape == (
    num_train,
    x_val.numel(),
)

assert attention_weights.shape == (
    num_train,
    x_val.numel(),
)

assert predictions.shape == (
    x_val.numel(),
)

torch.testing.assert_close(
    weight_sums,
    torch.ones_like(weight_sums),
)

print(
    "\nAttention Pooling validation passed."
)

Distance shape: (40, 50)
Similarity shape: (40, 50)
Attention Weight shape: (40, 50)
Prediction shape: (50,)

First five Weight sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

Attention Pooling validation passed.
